# ComfyUI on Colab — FLUX.2 [klein] Skin Enhancer

**Before you start:** Runtime → Change runtime type → **GPU (L4 or A100)**.

Run the cells **top to bottom**. The **last cell stays running** and prints a `https://....trycloudflare.com` link — that link is your ComfyUI GUI. Open it in a new tab.

If any cell shows a red error, paste it to Claude — some red text is harmless noise, some isn't.

In [ ]:
# 1) Install ComfyUI + ComfyUI-Manager
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
# ComfyUI-Manager lets you install the workflow's custom nodes from inside the GUI.
# If the URL below 404s, swap it for: https://github.com/ltdrdata/ComfyUI-Manager
!git clone https://github.com/Comfy-Org/ComfyUI-Manager /content/ComfyUI/custom_nodes/comfyui-manager

In [ ]:
# 2) Download FLUX.2 [klein] 4B models (Apache 2.0 — commercial-safe)
%cd /content/ComfyUI
!mkdir -p models/diffusion_models models/text_encoders models/vae

# Diffusion model (4B distilled, ~fast)
!wget -c "https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/diffusion_models/flux-2-klein-4b.safetensors" -O models/diffusion_models/flux-2-klein-4b.safetensors
# Text encoder (Qwen3 4B)
!wget -c "https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors" -O models/text_encoders/qwen_3_4b.safetensors
# VAE
!wget -c "https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/vae/flux2-vae.safetensors" -O models/vae/flux2-vae.safetensors

print('\nDone. Listing downloaded model files:')
!ls -lh models/diffusion_models models/text_encoders models/vae

In [ ]:
# 3) Install cloudflared (gives you a public link to the GUI)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# 4) Launch ComfyUI + print the GUI link. KEEP THIS CELL RUNNING.
# Look below for:  >>> OPEN THIS LINK FOR COMFYUI: https://xxxx.trycloudflare.com
import subprocess, threading, time, socket
%cd /content/ComfyUI

def tunnel(port):
    while True:
        time.sleep(0.5)
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if s.connect_ex(('127.0.0.1', port)) == 0:
            s.close(); break
        s.close()
    print('\nComfyUI is up — starting the public link...\n')
    p = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if 'trycloudflare.com' in l:
            print('>>> OPEN THIS LINK FOR COMFYUI:', l[l.find('https'):].strip())

threading.Thread(target=tunnel, daemon=True, args=(8188,)).start()
# --enable-cors-header (no value = '*') stops ComfyUI v19's host/origin check from 403-ing tunnel access.
!python main.py --dont-print-server --enable-cors-header